### Imports and global setup

In [18]:
## conda env: stereo_visionn
import os
import cv2
import glob
import numpy as np
from rtmlib import Wholebody, draw_skeleton, draw_bbox
from Util.util import BodyWithFeet, PoseTracker, Body, Custom, pose_to_bbox
import time
import json
from IPython.display import display, clear_output
from functools import partial

with open('./Util/models.json') as f:
    models = json.load(f)

#### Stock setup

In [ ]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
openpose_skeleton = False  # True for openpose-style, False for mmpose-style

body = Body(
    to_openpose=openpose_skeleton, mode="performance", backend=backend, device=device
)

#### Custom detector & model

In [24]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
detector_name = 'YOLOX_nano' # 'YOLOX_l_COCO','YOLOX_nano','YOLOX_tiny','YOLOX_s','YOLOX_m','YOLOX_l','YOLOX_x'
pose_name = 'RTMPose_x' # (26) 'RTMPose_t', 'RTMPose_s', 'RTMPose_m', 'RTMPose_l', 'RTMPose_m2', 'RTMPose_l2', 'RTMPose_x', (133) 'RTMW_l', 'RTMW_x'
kpt_labels = models['pose_models']["26"]["kpt_labels"]

custom = Custom(det_class='YOLOX',#'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose=models['pose_models']["26"]["models"][pose_name]['path'],
                pose_input_size=models['pose_models']["26"]["models"][pose_name]['input_size'],
                backend=backend,
                device=device) 

# pose_tracker = PoseTracker(custom,
#                            det_frequency=1,
#                            to_openpose=False,
#                            backend=backend, device=device)

load C:\Users\unger\.cache\rtmlib\hub\checkpoints\yolox_nano_8xb8-300e_humanart-40f6f0d0.onnx with onnxruntime backend
load C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.onnx with onnxruntime backend


### Mapping keypoints

In [ ]:
img = cv2.imread("./Util/person.png", cv2.IMREAD_COLOR)
# cv2.imshow("Image", img)
keypoints, scores = custom(img)
# keypoints, scores = body(img)

for index, point in enumerate(keypoints[0]):
    # if index<83 and index >= 66:

    cv2.circle(img, (round(point[0]), round(point[1])), 5, (0, 0, 255), 3)
    cv2.putText(
        img,
        f"{index}",
        (round(point[0] + 5), round(point[1]) + 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 100, 0),
        3,
    )
        # print(point, index)

cv2.imshow(
    "Image", cv2.resize(img, (round(img.shape[1] * 0.3), round(img.shape[0] * 0.3)))
)
# cv2.imwrite("./Util/Body_marker_locations_rtmpose_x_26.png", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### Video analysis

##### Plain and simple

In [26]:
input_folder = "./stereo_videos/validation_test"

for g in glob.glob(os.path.join(input_folder, "*.avi")):
    print(f'processing: {g}')
    file_name = g.split("\\")[-1]

    keypoints_over_time = []


    cap = cv2.VideoCapture(os.path.join(input_folder, file_name))

    if cap.isOpened() == False:
        print("Error opening video file")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()

    # Read until video is completed
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # inference
            keypoints, scores = custom(frame)

            # store results (frames * kpt * (x,y,confidence))
            current_values = np.squeeze(np.dstack((keypoints, scores)))

            # if multiple ppl detected, only keep the first one
            if current_values.shape != (26,3):
                current_values = current_values[0,:,:]
            keypoints_over_time.append(current_values)

            # img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.6)

            # img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.2)
            # boxes = [pose_to_bbox(x) for x in keypoints]
            # img_show = draw_bbox(frame, boxes, (0, 0, 255))

            ## check to see in ankle tracking switches when changing direction
            # cv2.circle(img_show,(round(keypoints[0][16][0]),round(keypoints[0][16][1])),5,(0,0,255),3 )

            # cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 10) == 0:

                fps=round(processed_num_frames/elapsed_time,2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio*60)


                display(f'exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes')
                display(f'done: {round(done_ratio*100,2)}%, fps: {fps}')

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break

    cap.release()
    cv2.destroyAllWindows()

    keypoints_over_time = np.asarray(keypoints_over_time[0:])
    print(f'shape of coordinates: {keypoints_over_time.shape}')

    out_file_name = os.path.join(input_folder,f'{file_name.split('.')[0]}')
    # np.save(
    #     out_file_name,
    #     keypoints_over_time,
    # )
    with open(out_file_name, 'w') as f:
        np.savez(out_file_name, pose_data=keypoints_over_time, kpt_labels=kpt_labels)
    print(f"Saved keypoint-coordinates to {out_file_name}")

shape of coordinates: (3908, 26, 3)
Saved keypoint-coordinates to ./stereo_videos/validation_test\43916681


In [27]:
keypoints_over_time.shape

(3908, 26, 3)

In [ ]:
np.load('./stereo_videos/validation_test\\43916681.npy').shape

##### Pose Tracker 

In [3]:
from functools import partial

# set up custom pose detection model
custom = partial(
    Custom,
    det_class='YOLOX',#'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose=models['pose_models'][pose_name]['path'],
                pose_input_size=models['pose_models'][pose_name]['input_size'],
                backend=backend,
                device=device,
)


# set up pose tracker to work with custom model
pose_tracker = PoseTracker(
    custom, det_frequency=10, to_openpose=False, backend=backend, device=device
)



input_folder = "./stereo_videos/validation_test"

for g in glob.glob(os.path.join(input_folder, "*.avi")):
    print(f"processing: {g}")
    file_name = g.split("\\")[-1]

    keypoints_over_time = []
    track_ids_over_time = []
    last_frame_track_ids_over_time = []

    cap = cv2.VideoCapture(os.path.join(input_folder, file_name))

    if cap.isOpened() == False:
        print("Error opening video file")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()

    # Read until video ends
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # inference
            # keypoints, scores = pose_tracker(frame)
            keypoints, scores, track_ids, last_track_ids = pose_tracker(frame)
            
            # shape result of current frame's inference to (kpt * (x,y,confidence))
            current_values = np.squeeze(np.dstack((keypoints, scores)))
            # store results (frames * kpt * (x,y,confidence))
            keypoints_over_time.append(current_values)
            track_ids_over_time.append(track_ids)
            last_frame_track_ids_over_time.append(last_track_ids)

            img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.6)

            img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.2)
            boxes = [pose_to_bbox(x) for x in keypoints]
            img_show = draw_bbox(frame, boxes, (0, 0, 255))

            ## check to see in ankle tracking switches when changing direction
            # cv2.circle(img_show,(round(keypoints[0][16][0]),round(keypoints[0][16][1])),5,(0,0,255),3 )

            cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 10) == 0:

                fps = round(processed_num_frames / elapsed_time, 2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio * 60)

                display(
                    f"exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes"
                )
                display(f"done: {round(done_ratio*100,2)}%, fps: {fps}, frame_num: {processed_num_frames}")

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break

    cap.release()
    cv2.destroyAllWindows()

    # keypoints_over_time = np.asarray(keypoints_over_time[0:])
    # print(f"shape of coordinates: {keypoints_over_time.shape}")

    # out_file_name = os.path.join(input_folder, f"{file_name.split('.')[0]}")
    # np.save(
    #     out_file_name,
    #     keypoints_over_time,
    # )
    # print(f"Saved keypoint-coordinates to {out_file_name}")

'exp_duration: 4.79 minutes, elapsed: 0.68 minutes'

'done: 14.25%, fps: 13.6, frame_num: 557'

In [ ]:
for current_ids, last_ids in zip(track_ids_over_time, last_frame_track_ids_over_time):
    print(current_ids, last_ids)

[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] []
[] [

: 

#### Save the coords of chosen keypoints

In [ ]:
out_folder_name = "./saved_coords"
# out_folder_name = input_folder
file_name = file_name.split('.')[0]
np.save(
    f"{os.path.join(out_folder_name, file_name)}_{detector_name}_{pose_name}.npy", keypoints_over_time
)